In [ ]:
import csv
import os
import pandas as pd
from mp_api.client import MPRester
from tqdm import tqdm
from google.colab import drive

# 1. Setup & Mount Drive
drive.mount('/content/drive')
API_KEY = "BUid9J786c6hqa8udYlaBgHZw8Utp016"
OUTPUT_FILE = "/content/drive/MyDrive/mp_data_batches.csv"
FIELDS = ["material_id", "formula_pretty", "band_gap", "is_stable", "energy_per_atom"]
BATCH_SIZE = 500  # Downloads 500 materials in one API call

with MPRester(API_KEY) as mpr:
    # 2. Get the master list of all IDs
    print("Fetching master list of Material IDs...")
    all_docs = mpr.materials.summary.search(fields=["material_id"])
    all_ids = [str(d.material_id) for d in all_docs]

    # 3. Check for existing progress to allow Resume
    existing_ids = set()
    if os.path.exists(OUTPUT_FILE):
        print("Checking existing file to resume...")
        try:
            # We only read the ID column to save RAM
            df_existing = pd.read_csv(OUTPUT_FILE, usecols=["material_id"])
            existing_ids = set(df_existing["material_id"].astype(str).tolist())
            print(f"Resuming: {len(existing_ids)} materials already finished.")
        except Exception:
            print("File found but empty or corrupted. Starting fresh.")

    # 4. Filter IDs
    ids_to_download = [m for m in all_ids if m not in existing_ids]
    total_to_do = len(ids_to_download)
    print(f"Remaining to download: {total_to_do}")

    # 5. Batch Processing Loop
    file_exists = os.path.isfile(OUTPUT_FILE)
    with open(OUTPUT_FILE, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if not file_exists:
            writer.writeheader()

        # Divide the ID list into chunks of BATCH_SIZE
        for i in tqdm(range(0, total_to_do, BATCH_SIZE)):
            batch_ids = ids_to_download[i : i + BATCH_SIZE]

            try:
                # Query the batch
                batch_docs = mpr.materials.summary.search(
                    material_ids=batch_ids,
                    fields=FIELDS
                )

                # Write the batch rows
                for doc in batch_docs:
                    row = {field: getattr(doc, field, None) for field in FIELDS}
                    writer.writerow(row)

                # Push to Drive immediately after each batch
                f.flush()
                os.fsync(f.fileno()) # Deep flush to ensure it hits the disk

            except KeyboardInterrupt:
                print("\nManual stop detected. Progress saved.")
                break
            except Exception as e:
                print(f"\nError in batch starting at {i}: {e}")
                continue

print(f"Process complete. Check your Drive for: {OUTPUT_FILE}")

In [ ]:
from mp_api.client import MPRester

API_KEY = "BUid9J786c6hqa8udYlaBgHZw8Utp016"

with MPRester(API_KEY) as mpr:
    # This retrieves the list of fields specifically for the 'summary' endpoint
    fields = mpr.materials.summary.available_fields

    print(f"Total fields available: {len(fields)}")
    print("---")
    for field in sorted(fields):
        print(field)

Total fields available: 69
---
band_gap
bandstructure
builder_meta
bulk_modulus
cbm
chemsys
composition
composition_reduced
database_IDs
decomposes_to
density
density_atomic
deprecated
deprecation_reasons
dos
dos_energy_down
dos_energy_up
e_electronic
e_ij_max
e_ionic
e_total
efermi
elements
energy_above_hull
energy_per_atom
equilibrium_reaction_energy_per_atom
es_source_calc_id
formation_energy_per_atom
formula_anonymous
formula_pretty
grain_boundaries
has_props
has_reconstructed
homogeneous_poisson
is_gap_direct
is_magnetic
is_metal
is_stable
last_updated
material_id
n
nelements
nsites
num_magnetic_sites
num_unique_magnetic_sites
ordering
origins
possible_species
property_name
shape_factor
shear_modulus
structure
surface_anisotropy
symmetry
task_ids
theoretical
total_magnetization
total_magnetization_normalized_formula_units
total_magnetization_normalized_vol
types_of_magnetic_species
uncorrected_energy_per_atom
universal_anisotropy
vbm
volume
warnings
weighted_surface_energy
weighte

In [ ]:
import pandas as pd
from mp_api.client import MPRester
import time

# 1. SETUP
API_KEY = "BUid9J786c6hqa8udYlaBgHZw8Utp016"
FIELDS = [
    "material_id", "formula_pretty", "nsites", "formation_energy_per_atom",
    "volume", "density", "band_gap", "symmetry"
]

data_list = []

# 2. INITIAL SEARCH & BATCH PROCESSING
with MPRester(API_KEY) as mpr:
    print("Searching for materials matching formulas...")
    all_docs = mpr.materials.summary.search(
        deprecated=False,
        formula=["ABC3", "A2BCD6","A2BC6", "ABCDE6","A2BCD3E3"],
        fields=["material_id"]
    )

    all_ids = [str(d.material_id) for d in all_docs]
    total_found = len(all_ids)
    print(f"Found {total_found} materials. Starting batch download...")

    # 3. BATCH PROCESSING LOOP (500 per request)
    batch_size = 500
    for i in range(0, total_found, batch_size):
        batch = all_ids[i : i + batch_size]
        print(f"Fetching batch {i//batch_size + 1} ({i} to {min(i + batch_size, total_found)})...")

        try:
            # Fetch detailed docs for the current batch
            docs = mpr.materials.summary.search(
                material_ids=batch,
                fields=FIELDS
            )

            for doc in docs:
                # Extract symmetry safely
                crystal_system = doc.symmetry.crystal_system if doc.symmetry else None

                data_list.append({
                    "material_id": str(doc.material_id),
                    "formula_pretty": doc.formula_pretty,
                    "nsites": doc.nsites,
                    "crystal_system": crystal_system,
                    "formation_energy_per_atom": doc.formation_energy_per_atom,
                    "volume": doc.volume,
                    "density": doc.density,
                    "band_gap": doc.band_gap
                })
        except Exception as e:
            print(f"Error at batch starting {i}: {e}")
            time.sleep(2)  # Short pause before retry if there's a connection issue
            continue

# 4. SAVE TO CSV
df = pd.DataFrame(data_list)
df.to_csv("mp_perovskites2.csv", index=False)
print(f"\n✅ Successfully saved {len(df)} rows to 'mp_perovskites.csv'")

Searching for materials matching formulas...


Retrieving SummaryDoc documents:   0%|          | 0/13243 [00:00<?, ?it/s]

Found 13243 materials. Starting batch download...
Fetching batch 1 (0 to 500)...


Retrieving SummaryDoc documents:   0%|          | 0/500 [00:00<?, ?it/s]

Fetching batch 2 (500 to 1000)...


...

Fetching batch 26 (12500 to 13000)...


Retrieving SummaryDoc documents:   0%|          | 0/500 [00:00<?, ?it/s]

Fetching batch 27 (13000 to 13243)...


Retrieving SummaryDoc documents:   0%|          | 0/243 [00:00<?, ?it/s]


✅ Successfully saved 13243 rows to 'mp_perovskites.csv'
